# 第六章 文本大数据基本分析方法
----
书籍名称：《大数据分析：Python 实践与大型语言模型应用》

谢佳松、刘娟



----

## 6.2 分词、预处理与词云图
### 6.2.2 如何进行中文分词

In [ ]:
dict_set = ["北京", "大学", "北京大学", "生","学生"]
def fmm_segment(text, dict_set):
    result = []
    while text:
        word = text[:]
        while word and word not in dict_set:
            word = word[:-1]
        if not word:
            word = text[0]
        result.append(word)
        text = text[len(word):]
    return result
text = "北京大学生"
segmented = fmm_segment(text, dict_set)
print(segmented)  


In [ ]:
def rmm_segment(text, dict_set):
    result = []
    while text:
        word = text[:]
        while word and word not in dict_set:
            word = word[1:]
        if not word:
            word = text[-1]
        result.insert(0, word)
        text = text[:-len(word)]
    return result
segmented = rmm_segment(text, dict_set)
print(segmented)  

### 6.2.3 使用 jieba 进行中文分词

In [ ]:
import jieba
print(jieba.__version__)  


In [ ]:
import jieba
text = jieba.lcut('大数据分析：Python 实践与大型语言模型应用')
print(text)

In [ ]:
text1 = jieba.lcut('大数据分析：Python 实践与大型语言模型应用',cut_all=True)
print(text1)

In [ ]:
text2 = jieba.lcut_for_search('大数据分析：Python 实践与大型语言模型应用')  # 搜索引擎模式
print(text2)

In [ ]:
import jieba.posseg as pseg
words = pseg.cut('大数据分析：Python 实践与大型语言模型应用')
for word, flag in words:
    print(f"{word} - {flag}")

In [ ]:
import jieba
text = '大数据分析：Python 实践与大型语言模型应用'
#导入自定义词典
jieba.load_userdict('dict/userdict.txt')   #直接导入jieba.load_userdict就可以，不需要再进行配置
new_seg = jieba.lcut(text)
new_seg #此时，"大型语言模型"不再拆分

In [ ]:
import re
import jieba
# 原始文本
text = "dddd好的ddd好的good?*\~访问"
# 使用正则表达式保留中文字符
clean_text = re.sub(r'[^\u4e00-\u9fa5]+', '', text)
# 对保留的中文文本进行分词
seg_list = jieba.lcut(clean_text)
# 打印分词结果
seg_list

### 6.2.4 文本预处理

In [ ]:
import re
import unicodedata

# 读取文件
with open('data/辽宁2024.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 步骤 1：去除多余空格和空行（包括中文空格 \u3000）
text = re.sub(r'\u3000|\xa0', '', text)        # 移除全角空格与不可见字符
text = re.sub(r'[ \t]+', ' ', text)            # 多余空格合并为一个
text = re.sub(r'\n\s*\n', '\n', text)          # 多个空行变为一个空行

# 步骤 2：统一全角符号为半角（如中文引号）
def fullwidth_to_halfwidth(s):
    return unicodedata.normalize('NFKC', s)

text = fullwidth_to_halfwidth(text)

# 步骤 3：保留关键中文标点“，”和“。”，删除其他无用特殊字符
# 可选：移除除常见中英文标点以外的字符（如 emoji、奇异符号）
text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9，,：:。！？；、“”‘’()\[\]{}《》<>]', '', text)

# 预览处理后文本（前500字）
print(text[:500])

In [ ]:
import jieba
# 加载停用词表
with open('dict/中文停用词表.txt', 'r', encoding='utf-8') as f:
    stopwords = set([line.strip() for line in f if line.strip()])
# 分词
words = jieba.lcut(text)
# 去除停用词和空白符
filtered_words = [word for word in words if word not in stopwords and word.strip()]
# 查看前50个有效词语
print(filtered_words[:50])

### 6.2.5 分词可视化工具：词云图

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import jieba
text = '''
乡村振兴是新时代的国家战略，全面推进农业农村现代化，
坚持农业农村优先发展，加快构建现代农业产业体系、
生产体系、经营体系。
'''
# 设置中文字体
font_path='font/Songti.ttc'
 
words = jieba.lcut(text)
processed_text = " ".join(words) #关键一步：使用jieba分词后，再拼接为空格分割的字符串
wordcloud = WordCloud(
    font_path=font_path,
    width=800,
    height=400,
    background_color='white'
).generate(processed_text)
 
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='nearest') #interpolation为插值算法函数
plt.axis('off')
plt.savefig("img/cloud1.pdf", format='pdf',dpi=600, bbox_inches='tight') 
plt.show()


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import jieba
import re
# 读取文件内容
f = open('data/辽宁2024.txt', 'r', encoding='utf-8').read()
#导入电脑字体
xs='font/Songti.ttc'
# 处理后的文本数据
wordlist = jieba.lcut(''.join(re.findall('[\u4e00-\u9fa5]+', f)))
# 加载中文停用词表
stop_words_file = open('dict/中文停用词表.txt', 'r', encoding='utf-8')
stop_words = set(stop_words_file.read().splitlines())
stop_words_file.close()
# 过滤停用词
wordlist = [word for word in wordlist if word not in stop_words]
# 将处理后的文本数据传递给 WordCloud 对象的 generate 方法
#如果你正在使用 wordcloud 库，可以使用以下方式在Jupyter Notebook中显示词云图：
wordcloud = WordCloud(
    width=1000,
    height=700,
    mode="RGBA",
    background_color='white',
    colormap='rainbow',
    font_path=xs
).generate(' '.join(wordlist))  # 注意这里将列表转换为空格分隔的字符串
# 在Jupyter Notebook中显示词云图
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.savefig("img/cloud2.pdf", format='pdf',dpi=600, bbox_inches='tight') 
plt.show()

## 6.3 基于词典的文本分析
### 6.3.1 词典法基本概念与框架

In [ ]:
import jieba
import pandas as pd
#创新词典
inovation_words = ['创新', '科技', '研发', '高校', '技术', '科学', '理论', '专利', '攻克', '改良', '工艺', '前沿', '尖端']
#环保词典
green_words = ['绿色', '节能', '低碳', '环保', '环境友好', '无污染']
#短视主义词典
push_words = ['加快', '尽快', '抓紧', '月底', '年底', '争取', '马上', '立刻', '年内', '数月', '数年']
#模棱两可词典
prob_words = ['可能', '大概', '左右', '估计', '大约']
#定义函数
def analysis_info(text):
    """
    对文本进行分词后，统计四类词典中出现的词频数量。
    返回结果为包含各类词频的 Pandas Series。
    """
    words = jieba.lcut(text)
    
    # 分别统计每类词频
    res = {
        'total_words': len(words),  # 分词总数
        'inovation': sum(words.count(w) for w in inovation_words),
        'green': sum(words.count(w) for w in green_words),
        'push': sum(words.count(w) for w in push_words),
        'prob': sum(words.count(w) for w in prob_words)
    }
    return pd.Series(res)
#词袋模型分析
analysis_info(text='2025年公司规范运作，坚持科技创新，保持持续发展，同时加快技术改良和绿色环保投入，争取年内完成研发。')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import jieba
import pandas as pd
from matplotlib.font_manager import FontProperties
font_path = "font/Songti.ttc"
songti = FontProperties(fname=font_path)

inovation_words = ['创新', '科技', '研发', '高校', '技术', '科学', '理论', '专利', '攻克', '改良', '工艺', '前沿', '尖端']
green_words = ['绿色', '节能', '低碳', '环保', '环境友好', '无污染']
push_words = ['加快', '尽快', '抓紧', '月底', '年底', '争取', '马上', '立刻', '年内', '数月', '数年']
prob_words = ['可能', '大概', '左右', '估计', '大约']

# 分析函数
def analysis_info(text):
    words = jieba.lcut(text)
    res = {
        '创新': sum(words.count(w) for w in inovation_words),
        '环保': sum(words.count(w) for w in green_words),
        '短视': sum(words.count(w) for w in push_words),
        '模糊表达': sum(words.count(w) for w in prob_words)
    }
    return pd.Series(res)

# 示例文本
text = "2022年是公司规范运作，坚持科技创新，保持持续发展，同时加快技术改良和绿色环保投入，争取年内完成研发。"
scores = analysis_info(text)

# 雷达图绘制
labels = list(scores.index)
data = scores.values
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
data = np.concatenate((data, [data[0]]))
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, data, color='teal', linewidth=2)
ax.fill(angles, data, color='teal', alpha=0.3)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontproperties=songti, fontsize=12)
ax.set_title("文本关键词频率雷达图", fontproperties=songti, fontsize=14, pad=20)
ax.grid(True)
plt.show()

### 6.3.2 实战案例：上市公司MD&A中的创新表述

In [ ]:
innovation_words = ["创新", "研发", "技术", "数字化", "智能化", "人工智能"]

def analyse(text):
    stop_words = set(line.strip() for line in open("dict/中文停用词表.txt", "r", encoding="utf-8"))
    words = [w for w in jieba.lcut(text) if w not in stop_words and len(w) > 1]
    total_words = len(words)
    innovation_count = sum(words.count(w) for w in innovation_words)
    innovation_ratio = innovation_count / total_words if total_words > 0 else 0
    return pd.Series({
        "total_words": total_words,
        "innovation_count": innovation_count,
        "innovation_ratio": innovation_ratio
    })


In [ ]:
df = pd.read_excel('data/MDA.xlsx')
df = df[["年份", "公司代码", "内容"]]


In [ ]:
import time
start_time = time.time()
info_df = df['内容'].apply(analyse)
# 合并原始数据与分析结果
res_df = pd.concat([df, info_df], axis=1)
end_time = time.time()
print("Execution time: {:.2f} seconds".format(end_time - start_time))
# 预览结果
res_df.head()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
font_path = 'font/Songti.ttc'
my_font = fm.FontProperties(fname=font_path)
# 按年份分组，计算每年创新占比均值
yearly_ratio = res_df.groupby('年份')['innovation_ratio'].mean().reset_index()
# 绘图
plt.figure(figsize=(10, 6))
plt.plot(yearly_ratio['年份'], yearly_ratio['innovation_ratio'], marker='o', linestyle='-', color='royalblue')
plt.title('不同年份上市公司MD&A中“创新”表述占比趋势', fontproperties=my_font, fontsize=16)
plt.xlabel('年份', fontproperties=my_font, fontsize=14)
plt.ylabel('创新词占比（innovation_ratio）', fontproperties=my_font, fontsize=14)
plt.xticks(yearly_ratio['年份'], rotation=45, fontproperties=my_font)
plt.yticks(fontproperties=my_font)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.font_manager as fm

font_path = "font/Songti.ttc"
my_font = fm.FontProperties(fname=font_path)
plt.figure(figsize=(12, 6))
try:
    import seaborn as sns
    sns.boxplot(data=res_df, x="年份", y="innovation_ratio", palette="Blues")
except Exception:
    groups = [g["innovation_ratio"].dropna().values for _, g in res_df.groupby("年份")]
    labels = list(res_df.groupby("年份").groups.keys())
    plt.boxplot(groups, labels=labels, patch_artist=True)
plt.title("各年份上市公司MD&A文本中创新表述比例的箱线图", fontproperties=my_font, fontsize=14)
plt.xlabel("年份", fontproperties=my_font, fontsize=12)
plt.ylabel("创新表述占比", fontproperties=my_font, fontsize=12)
plt.xticks(fontproperties=my_font)
plt.yticks(fontproperties=my_font)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.font_manager as fm

font_path = "font/Songti.ttc"
my_font = fm.FontProperties(fname=font_path)
res_df["年份"] = res_df["年份"].astype(str)
plt.figure(figsize=(12, 8))
try:
    from joypy import joyplot
    joyplot(data=res_df, by="年份", column="innovation_ratio", figsize=(12, 8), colormap=plt.cm.Blues, fade=True, linewidth=1)
except Exception:
    for year, grp in res_df.groupby("年份"):
        grp["innovation_ratio"].plot(kind="density", label=year)
    plt.legend(prop=my_font)
plt.title("各年份MD&A文本中创新占比的密度分布（峰峦图）", fontproperties=my_font, fontsize=14)
plt.xlabel("创新占比", fontproperties=my_font, fontsize=12)
plt.ylabel("年份", fontproperties=my_font, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
font_path = "font/Songti.ttc"
my_font = fm.FontProperties(fname=font_path)
pivot_data = res_df.pivot_table(index="公司代码", columns="年份", values="innovation_ratio")
plt.figure(figsize=(12, 8))
try:
    import seaborn as sns
    sns.heatmap(pivot_data, cmap="YlOrBr", linewidths=0.5, linecolor="white", cbar_kws={"label": "创新词占比"}, annot=False)
except Exception:
    plt.imshow(pivot_data.fillna(0).values, aspect="auto", cmap="YlOrBr")
    plt.colorbar(label="创新词占比")
plt.title("公司-年份创新占比热力图", fontproperties=my_font, fontsize=16)
plt.xlabel("年份", fontproperties=my_font, fontsize=14)
plt.ylabel("公司代码", fontproperties=my_font, fontsize=14)
plt.xticks(range(len(pivot_data.columns)), pivot_data.columns, rotation=45)
plt.tight_layout()
plt.show()


### 6.3.4 词语权重

In [ ]:
#1.分词（英文直接以空格分词）
corpus = ['this is the first document',
        'this is the second second document',
        'and the third one',
        'is this the first document']
words_list = list()
#遍历 corpus 列表中的每一项，对第 i 篇文本执行按空格分词，添加进 words_list
for i in range(len(corpus)):
    words_list.append(corpus[i].split(' '))
print(words_list)

In [ ]:
from collections import Counter
count_list = list()
for i in range(len(words_list)):
    count = Counter(words_list[i])
    count_list.append(count)
print(count_list)


In [ ]:
import math
def tf(word, count):
    return count[word] / sum(count.values())


def idf(word, count_list):
    n_contain = sum([1 for count in count_list if word in count])
    return math.log(len(count_list) / (1 + n_contain))


def tf_idf(word, count, count_list):
    return tf(word, count) * idf(word, count_list)

In [ ]:
for i, count in enumerate(count_list):
    print("第 {} 个文档 TF-IDF 统计信息".format(i + 1))
    scores = {word : tf_idf(word, count, count_list) for word in count}
    sorted_word = sorted(scores.items(), key = lambda x : x[1], reverse=True)
    for word, score in sorted_word:
        print("\tword: {}, TF-IDF: {}".format(word, round(score, 5)))

In [ ]:
from collections import Counter
import math
import jieba
# 样本文本
text = "公司2025年实现10%收入增长，但面临供应链挑战和通货膨胀压力"
words = jieba.lcut(text)
word_freq = Counter(words)
# 假设语料库有1000篇文档，包含词语的文档数
doc_total = 1000
doc_with_word = {"增长": 200, "挑战": 150, "通货膨胀": 50}
# 计算TF-IDF
tf_idf_scores = {}
for word in word_freq:
    tf = word_freq[word] / len(words)
    idf = math.log(doc_total / (doc_with_word.get(word, 1) + 1))
    tf_idf = tf * idf
    tf_idf_scores[word] = tf_idf
for word, score in tf_idf_scores.items():
    print(f"词语: {word}, TF-IDF 权重: {score:.4f}")
# 假设领域权重
domain_weights = {"增长": 0.8, "挑战": 0.6, "通货膨胀": 0.9}
for word in tf_idf_scores:
    final_weight = tf_idf_scores[word] * domain_weights.get(word, 0.5)
    print(f"词语: {word}, 最终权重: {final_weight:.4f}")

### 6.3.5 利用N-gram捕捉上下文语义特征

In [ ]:
import jieba
from collections import Counter

# 样本文本
text = "公司实现显著的利润增长，但市场不呈现明显复苏迹象"
words = jieba.lcut(text)

# 生成 N-gram (N=2 和 N=3)
def generate_ngrams(words, n):
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append(" ".join(words[i:i + n]))
    return ngrams

bigrams = generate_ngrams(words, 2)
trigrams = generate_ngrams(words, 3)

# 统计 N-gram 频率
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

# 预定义修饰词和否定词词典
modifier_dict = {"显著的": 0.7, "明显": 0.6, "轻微": 0.3}
negation_dict = {"不": -1, "不是": -1}

# 识别并调整权重
for bigram in bigram_freq:
    words = bigram.split()
    if words[0] in negation_dict:
        print(f"否定模式: {bigram}, 权重调整: {negation_dict[words[0]]}")
    if words[1] in modifier_dict:
        print(f"强度修饰: {bigram}, 权重因子: {modifier_dict[words[1]]}")

for trigram in trigram_freq:
    words = trigram.split()
    if words[0] in negation_dict and words[2] in modifier_dict:
        print(f"否定+强度模式: {trigram}, 综合权重: {negation_dict[words[0]] * modifier_dict[words[2]]}")

In [ ]:
import jieba
from collections import Counter
import math
# 设定样本文本（包含多种问题场景）
text = "公司实现显著的利润增长，但市场不呈现明显复苏迹象。公司面临挑战，但最终实现增长。"
text1 = "增长不明显"
text2 = "不明显增长"
text_error = "公司实现利润增长率"
words = jieba.lcut(text)  #精确分词
# 函数：生成 N-gram
def generate_ngrams(words, n):
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append(" ".join(words[i:i + n]))
    return ngrams


In [ ]:
words1 = jieba.lcut(text1)
words2 = jieba.lcut(text2)
bigrams1 = generate_ngrams(words1, 2)
bigrams2 = generate_ngrams(words2, 2)
print("语义理解能力对比：")
print(f"Text1: {text1}, Bigram: {bigrams1}, 语义焦点: 增长受限")
print(f"Text2: {text2}, Bigram: {bigrams2}, 语义焦点: 增长不明显（相同输出，失真）")


In [ ]:
trigrams = generate_ngrams(words, 3)
trigram_freq = Counter(trigrams)
print("\n数据稀疏性严重（Trigram 频次）：")
for trigram, freq in trigram_freq.items():
    print(f"{trigram}: {freq}")


In [ ]:
bigrams = generate_ngrams(words, 2)
print("\n不能捕捉长距离依赖（Bigram）：")
for bigram in bigrams:
    if "挑战" in bigram or "增长" in bigram:
        print(f"{bigram}")


In [ ]:
words_error = jieba.lcut(text_error)  # 默认分词可能出错
bigrams_error = generate_ngrams(words_error, 2)
print("\n过度依赖分词精度（错误分词）：")
print(f"原始文本: {text_error}, 分词: {words_error}, Bigram: {bigrams_error}")
print("优化建议：使用自定义词典（如'利润增长率'）")


In [ ]:
trigrams_full = generate_ngrams(words, 3)
fourgrams = generate_ngrams(words, 4)
print("\nN 取值过大导致语义误识别：")
print("Trigram (合理):", [t for t in trigrams_full if "利润" in t])
print("Fourgram (误导):", [f for f in fourgrams if "公司" in f])
print("结论：N=4 可能生成'公司 实现 利润 增长'以外的伪关联")


## 6.4 词向量和词嵌入
### 6.4.1 词向量基础：One-hot编码

In [ ]:
import jieba
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 样本文本
text = "公司2025年实现增长"
words = jieba.lcut(text)  #jieba精确分词
unique_words = list(set(words))  # 去重
# 编码
label_encoder = LabelEncoder()
integer_encoded = label_encoder.fit_transform(unique_words)
onehot_encoder = OneHotEncoder()
onehot_encoded = onehot_encoder.fit_transform(integer_encoded.reshape(-1, 1))
# 输出结果
print("词汇表:", unique_words)
print("One-Hot 编码:", onehot_encoded)

### 6.4.2 使用神经网络方法进行词嵌入

In [ ]:
import jieba
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np

# 样本文本
text = "公司2025年实现增长"
# 1. 分词
words = list(jieba.cut(text)) #jieba精确分词
print("分词结果：", words)
# 2. 整数编码
label_encoder = LabelEncoder()
word_labels = label_encoder.fit_transform(words)
print("整数编码：", word_labels)
print("对应词表：", list(label_encoder.classes_))
# 3. One-Hot编码
onehot_encoder = OneHotEncoder(sparse_output=False)  # 改为 sparse_output=False
word_labels = word_labels.reshape(len(word_labels), 1)
onehot_vectors = onehot_encoder.fit_transform(word_labels)
print("One-Hot 编码：\n", onehot_vectors)
# 4. 格式化展示
for word, vector in zip(words, onehot_vectors):
    print(f"{word} -> {vector}")

In [ ]:
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

df = pd.read_csv('data/md&a10000.csv')
df.drop(columns=['Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'], inplace=True)
df.head()

In [ ]:
import time
start_time = time.time()
# 定义一个函数来准备文本数据
def prepare_text_data(df_column):
#simple_preprocess 函数来自于 Gensim 库，它用于将文本转换为简单的预处理后的单词列表
    #注：这是为便于展示，实际上为了准确，还需要严谨的分词，譬如使用jieba进行分词
    return df_column.apply(lambda x: simple_preprocess(x))
# 准备文本数据
text_data = prepare_text_data(df['经营讨论与分析内容'])
# 训练 Word2Vec 模型
word2vec_model = Word2Vec(sentences=text_data, vector_size=100, window=2, min_count=1, sg=1)
#计算运算时间
end_time = time.time()
execution_time = end_time - start_time
print("Execution time: {:.2f} seconds".format(execution_time))

In [ ]:
word = "发展"
vector = word2vec_model.wv[word]
print("发展", ":", vector)

In [ ]:
similar_words = word2vec_model.wv.most_similar("发展", topn=10)
print(similar_words)

### 6.4.3 论文复现：基于Word2vec模型扩展人工智能词典

In [ ]:
import gensim
model = 'model/wiki.zh.text.model'
model = gensim.models.Word2Vec.load(model)
result = model.wv.most_similar(u"北京")   #u 前缀用于表示这是一个 Unicode 字符串
for e in result:    #e[0] 是与指定词语最相似的词语，e[1] 是相似度分数
    print(e[0], e[1])

In [ ]:
import gensim
model = 'model/wiki.zh.text.model'
model = gensim.models.Word2Vec.load(model)
result = model.wv.most_similar(u"大数据")  
for e in result:    
    print(e[0], e[1])

In [ ]:
import gensim
import pandas as pd

# 加载利用中文维基百科文本语料训练好的Word2Vec模型
model = 'model/wiki.zh.text.model'
model = gensim.models.Word2Vec.load(model)

# 定义种子词列表
seed_words = [
    "人工智能", "计算机视觉", "AI产品", "人机交互", "AI芯片",
    "深度学习", "机器翻译", "机器学习", "神经网络", "生物识别",
    "图像识别", "数据挖掘", "特征识别", "语音合成", "语音识别",
    "知识图谱", "智慧银行", "智能保险", "人机协同", "智能监管",
    "智能教育", "智能客服", "智能零售", "智能农业", "智能投顾",
    "增强现实", "虚拟现实", "智能医疗", "智能音箱", "智能语音",
    "智能政务", "自动驾驶", "智能运输", "卷积神经网络", "声纹识别",
    "特征提取", "无人驾驶", "智能家居", "问答系统", "人脸识别",
    "商业智能", "智慧金融", "循环神经网络", "强化学习", "智能体",
    "智能养老", "大数据营销", "大数据风控", "大数据分析", "大数据处理",
    "支持向量机", "长短期记忆", "机器人流程自动化", "自然语言处理", "分布式计算",
    "知识表示", "智能芯片", "可穿戴产品", "大数据管理", "智能传感器",
    "模式识别", "边缘计算", "大数据平台", "智能计算", "智能搜索",
    "物联网", "云计算", "增强智能", "语音交互", "智能环保",
    "人机对话", "深度神经网络", "大数据运营"
]
#存储结果
result_df = pd.DataFrame(columns=["Seed_Word", "Similar_Word"])

# 对每个种子词进行处理
for seed_word in seed_words:
    try:
        # 获取与种子词最相近的前10个词
        similar_words = [word for word, _ in model.wv.most_similar(seed_word, topn=10)]
        # 将结果添加到DataFrame中
        result_df = pd.concat([result_df, pd.DataFrame({"Seed_Word": [seed_word]*10, "Similar_Word": similar_words})], ignore_index=True)
    except KeyError:
        # 如果种子词不在词汇表中，则跳过
        pass

# 将结果保存为CSV文件
result_csv_path = "data/similar_words.csv"
result_df.to_csv(result_csv_path, index=False)
print("结果已保存至:", result_csv_path)

In [ ]:
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = 'font/Songti.ttc'
my_font = fm.FontProperties(fname=font_path)
file_path = 'data/similar_words.csv'
df = pd.read_csv(file_path)

# 获取前 9 个独特的种子词
seed_words = df['Seed_Word'].unique()[:9]

# 创建 3x3 子图
fig, axes = plt.subplots(3, 3, figsize=(16, 16))
axes = axes.flatten()

for i, seed_word in enumerate(seed_words):
    subset = df[df['Seed_Word'] == seed_word]['Similar_Word']
    word_freq = subset.value_counts().to_dict()

    # 生成词云
    wc = WordCloud(
        font_path=font_path,
        background_color='white',
        width=400,
        height=400
    ).generate_from_frequencies(word_freq)

    # 绘制到子图
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(f"'{seed_word}' 相似词词云", fontproperties=my_font, fontsize=14)
    axes[i].axis('off')
for j in range(len(seed_words), 9):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig("img/AIword2Vec.pdf", format='pdf', bbox_inches='tight') 
plt.show()

### 6.4.4 进阶词嵌入技术：GloVe、BERT和GPT

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations
import jieba
import math

# ------------------------------
# 1. 样本文本数据
# ------------------------------
corpus = [
    "公司实现创新增长",
    "经济增长需要技术创新",
    "市场 竞推动企业发展",
    "企业创新技术研发",
    "经济复苏依赖政策和创新"
]

# ------------------------------
# 2. 分词与词典构建
# ------------------------------
tokenized_corpus = [list(jieba.lcut(sentence)) for sentence in corpus]
print("分词结果：", tokenized_corpus)

# 生成词典
vocab = list(set(word for sentence in tokenized_corpus for word in sentence))
vocab_size = len(vocab)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
print("词典：", vocab)

# ------------------------------
# 3. 构建共现矩阵
# ------------------------------
window_size = 2  # 滑动窗口大小
co_occurrence = np.zeros((vocab_size, vocab_size))

for sentence in tokenized_corpus:
    for idx, word in enumerate(sentence):
        word_idx = word2idx[word]
        start = max(idx - window_size, 0)
        end = min(idx + window_size + 1, len(sentence))
        for i in range(start, end):
            if i != idx:
                context_word_idx = word2idx[sentence[i]]
                co_occurrence[word_idx, context_word_idx] += 1

print("共现矩阵：\n", pd.DataFrame(co_occurrence, index=vocab, columns=vocab))

# ------------------------------
# 4. 定义GloVe模型训练
# ------------------------------
embedding_dim = 10  # 词向量维度
W = np.random.rand(vocab_size, embedding_dim)  # 主词向量
C = np.random.rand(vocab_size, embedding_dim)  # 上下文词向量
bias_W = np.zeros(vocab_size)
bias_C = np.zeros(vocab_size)

def weight_func(x, x_max=100, alpha=0.75):
    return (x / x_max) ** alpha if x < x_max else 1

learning_rate = 0.05
epochs = 100

for epoch in range(epochs):
    total_loss = 0
    for i in range(vocab_size):
        for j in range(vocab_size):
            x_ij = co_occurrence[i][j]
            if x_ij > 0:
                weight = weight_func(x_ij)
                log_x_ij = np.log(x_ij)
                diff = np.dot(W[i], C[j]) + bias_W[i] + bias_C[j] - log_x_ij
                loss = weight * (diff ** 2)
                total_loss += loss
                
                # 梯度下降更新
                grad = 2 * weight * diff
                W[i] -= learning_rate * grad * C[j]
                C[j] -= learning_rate * grad * W[i]
                bias_W[i] -= learning_rate * grad
                bias_C[j] -= learning_rate * grad
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}")

# ------------------------------
# 5. 词向量查询
# ------------------------------
word_vectors = W + C  # 最终词向量
def find_similar_words(word, top_n=3):
    if word not in word2idx:
        return []
    idx = word2idx[word]
    target_vec = word_vectors[idx]
    similarities = []
    for i, other_word in enumerate(vocab):
        if other_word != word:
            sim = np.dot(target_vec, word_vectors[i]) / (np.linalg.norm(target_vec) * np.linalg.norm(word_vectors[i]))
            similarities.append((other_word, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("与 '创新' 最相似的词：", find_similar_words("创新"))

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# 1. 加载预训练的 BERT 模型和分词器
model_name = "bert-base-chinese"  # 使用中文 BERT 模型
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
# 2. 输入文本并进行分词
text = "公司实现创新增长"
tokens = tokenizer.tokenize(text)   # 将句子拆分为 BERT 的子词单元
print("分词结果：", tokens)
# 将文本转换为模型输入格式
inputs = tokenizer(text, return_tensors="pt")  # 输出 PyTorch 张量
print("Token IDs：", inputs['input_ids'])
# 3. 获取 BERT 的词向量
with torch.no_grad():  # 不进行梯度计算
    outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state  # 最后一层隐层输出
    cls_embedding = last_hidden_state[:, 0, :]     # [CLS] 向量
    print("句子向量维度：", cls_embedding.shape)
    print("句子向量：", cls_embedding)

# 4. 提取每个 Token 的词向量
for token, vec in zip(tokens, last_hidden_state[0][1:len(tokens)+1]):  # [CLS]除外
    print(f"Token: {token}, 向量维度: {vec.shape}")

In [ ]:
from transformers import GPT2Tokenizer, GPT2Model
import torch

# ------------------------------
# 1. 加载 GPT2 模型和分词器
# ------------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2")

# 设置为评估模式（非训练）
model.eval()

# ------------------------------
# 2. 定义输入文本
# ------------------------------
text = "Artificial Intelligence boosts innovation"

# 对文本进行分词并编码为 ID
inputs = tokenizer(text, return_tensors="pt")

# ------------------------------
# 3. 获取词向量
# ------------------------------
with torch.no_grad():
    outputs = model(**inputs)
    # 最后一层隐状态 (batch_size, sequence_length, hidden_size)
    last_hidden_states = outputs.last_hidden_state

# ------------------------------
# 4. 输出词向量结果
# ------------------------------
for token, vector in zip(tokenizer.tokenize(text), last_hidden_states[0]):
    print(f"词语: {token}, 词向量维度: {vector.shape}, 示例向量值前5项: {vector[:5]}")

### 6.4.5 词嵌入前沿：Ada-002

In [ ]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
# 指定API路径
env_file_path = "OPENAI_API_KEY.env"
# 加载环境变量文件
load_dotenv(env_file_path)
def get_openai_key():
    _ = load_dotenv(find_dotenv())
    return os.environ['OPENAI_API_KEY']
openai.api_key = get_openai_key()

from langchain.embeddings.openai import OpenAIEmbeddings 
embedding = OpenAIEmbeddings()
print("Model:", embedding.model)

In [ ]:
sentence1_chinese = "东北财经大学" 
sentence2_chinese = "中央财经大学" 

embedding1_chinese = embedding.embed_query(sentence1_chinese) 
embedding2_chinese = embedding.embed_query(sentence2_chinese) 
embedding1_chinese